In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [2]:
torch.manual_seed(42)

# Study Method:
# Online  → 0
# Offline → 1
# Hybrid  → 2

# Subject:
# Math      → 0
# Physics   → 1
# Chemistry → 2

# Difficulty:
# Easy   → 0
# Medium → 1
# Hard   → 2

n = 1000

# Categorical features
study_method = torch.randint(0, 3, (n,))
subject = torch.randint(0, 3, (n,))
difficulty = torch.randint(0, 3, (n,))

# Numerical features
hours = torch.rand(n) * 8
previous_score = torch.rand(n) * 60 + 40

# Normalisation of hours
hours_mean = hours.mean()
hours_std = hours.std()

hours_n = (hours - hours_mean) / hours_std

# Normalisation of previous scores
ps_mean = previous_score.mean()
ps_std = previous_score.std()

ps_n = (previous_score - ps_mean) / ps_std


y_real = (
    40
    + hours_n * 4
    + ps_n * 0.4
    + study_method.float() * 3
    + subject.float() * 2
    - difficulty.float() * 4
)

# Add a little noise
y_real += torch.randn(n) * 2

In [3]:
from torch.utils.data import TensorDataset

X = torch.stack([study_method, subject, difficulty, hours_n, ps_n], dim = 1)
ds = TensorDataset(X, y_real.unsqueeze(1))

In [4]:
x_sample, y_sample = ds[0]
x_sample
y_sample


tensor([40.4139])

In [5]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Load the embedding functions
        self.study_emb = nn.Embedding(3, 2) # Three categorues and two out array size
        self.subject_emb = nn.Embedding(3, 2)
        self.difficulty_emb = nn.Embedding(3, 2)

        self.Layers = nn.Sequential(
            nn.Linear(2 + 2 + 2 + 1 + 1, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        # Extracting all the categorical data

        study = x[: ,0].long() # .long() makes it an integer
        subject = x[: ,1].long()
        difficulty = x[:, 2].long()

        # Extracting all the Numberical Data

        hours = x[:, 3].unsqueeze(1) # bcz they need to be [[1], [2], [3]] not [1, 2, 3]
        previous = x[:, 4].unsqueeze(1)

        study_vec = self.study_emb(study)
        subject_vec = self.subject_emb(subject)
        difficulty_vec = self.difficulty_emb(difficulty)

        x = torch.cat([
            study_vec,
            subject_vec,
            difficulty_vec,
            hours,
            previous
        ], dim = 1)

        # Make prediction
        return self.Layers(x)

In [6]:
from torch.utils.data import DataLoader

loader = DataLoader(
    ds,
    batch_size = 128,
    shuffle=True
)

In [ ]:
model = MyModel()
loss_fn = nn.MSELoss()
loss_history = []
optimiser = optim.Adam(model.parameters(), 0.01)
loss = torch.tensor(float("inf"))
traversal_history = []

In [13]:
accuracy = 0.1
epoachs = 50
i = 0
j = 0
for epoch in range(epoachs):

    for X_sample, y_sample in loader:

        # Reset Gradients
        optimiser.zero_grad()

        # Forward Pass
        y_pred = model(X_sample)

        # loss
        loss = loss_fn(y_pred, y_sample)
        loss_history.append(loss.item())

        # Backward
        loss.backward()

        # Change Weights and biases
        optimiser.step()

        # For Seaborn
        j += 1

    i += 1
    traversal_history.append(j)
    print(f"Traversal {i} : loss = {loss.item()}")



Traversal 1 : loss = 371.7947692871094
Traversal 2 : loss = 66.92893981933594
Traversal 3 : loss = 46.612060546875
Traversal 4 : loss = 12.04796028137207
Traversal 5 : loss = 11.448440551757812
Traversal 6 : loss = 7.720290660858154
Traversal 7 : loss = 5.078403949737549
Traversal 8 : loss = 4.94519567489624
Traversal 9 : loss = 4.519164085388184
Traversal 10 : loss = 4.625063419342041
Traversal 11 : loss = 4.097383499145508
Traversal 12 : loss = 4.410199165344238
Traversal 13 : loss = 3.9558217525482178
Traversal 14 : loss = 2.91515851020813
Traversal 15 : loss = 4.823326110839844
Traversal 16 : loss = 4.444930076599121
Traversal 17 : loss = 3.8108530044555664
Traversal 18 : loss = 4.864831924438477
Traversal 19 : loss = 4.064248085021973
Traversal 20 : loss = 4.544351577758789
Traversal 21 : loss = 5.309108257293701
Traversal 22 : loss = 4.956738471984863
Traversal 23 : loss = 4.673948764801025
Traversal 24 : loss = 4.904357433319092
Traversal 25 : loss = 4.67189359664917
Traversal 2

How to Interpret Gradient Statistics  
Gradient Magnitude (Scale):
Healthy Range: Usually between $0.001$ and $1.0$ (or up to single digits/low tens depending on learning rate and loss scale).Near 0 (e.g., $< 10^{-5}$):  
Vanishing gradient. The layer isn't learning.Huge (e.g., $> 100$ or huge spikes):   
Exploding gradient. Updates will overshoot and cause unstable training.  
Mean vs. Max:  
Mean: Tells you the overall direction and magnitude of updates across all parameters in that layer.  
Max: Shows if any single parameter/weight is experiencing extreme spikes.

In [9]:
for name, param in model.named_parameters():
    if param.grad is not None:
        print(
            name,
            "Grad Mean : ", param.grad.mean().item(), "   ",
            "Grad Max : ", param.grad.max().item()
        )

study_emb.weight Grad Mean :  10.263856887817383     Grad Max :  77.24972534179688
subject_emb.weight Grad Mean :  41.15544891357422     Grad Max :  55.740116119384766
difficulty_emb.weight Grad Mean :  -5.631845951080322     Grad Max :  60.801021575927734
Layers.0.weight Grad Mean :  0.691862940788269     Grad Max :  24.04027557373047
Layers.0.bias Grad Mean :  -12.954318046569824     Grad Max :  0.05079883337020874
Layers.2.weight Grad Mean :  -66.28826904296875     Grad Max :  0.0
Layers.2.bias Grad Mean :  -36.818809509277344     Grad Max :  -36.818809509277344
